In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import files
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# 1. Cargar datos y aplicar regla de negocio (solo adeudos disponibles para cobro)
archive = files.upload()
df = pd.read_csv('dataset_cobranza_train.csv')
df_activo = df[df['situacion_adeudo'] == 1].copy()

In [ ]:
print("Dimensines", df.shape)
display(df.head())

In [ ]:
# Ingeniería de características rápida: porcentaje ya abonado
df_activo['porcentaje_abonado'] = df_activo['pagos_parciales'] / (
    df_activo['monto_pendiente'] + df_activo['pagos_parciales']
)

features = [
    'monto_pendiente',
    'dias_vencimiento',
    'incumplimientos_anteriores',
    'pagos_parciales',
    'porcentaje_abonado',
    'intentos_contacto',
    'respuesta_contacto',
]

X = df_activo[features]
y = df_activo['recuperado']

# División 80% entrenamiento, 20% prueba (estratificada por el desbalanceo 74/26)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [ ]:
# 2. Preprocesamiento: OneHot para respuesta_contacto, StandardScaler para numéricas
num_cols = [
    'monto_pendiente',
    'dias_vencimiento',
    'incumplimientos_anteriores',
    'pagos_parciales',
    'porcentaje_abonado',
    'intentos_contacto',
]
cat_cols = ['respuesta_contacto']

preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols),
    ]
)

In [ ]:
# 3. Modelo Baseline (Visto en clase: Naive Bayes)
pipe_nb = Pipeline([('prep', preprocesador), ('clf', GaussianNB())])
pipe_nb.fit(X_train, y_train)
preds_nb = pipe_nb.predict(X_test)
probs_nb = pipe_nb.predict_proba(X_test)[:, 1]

In [ ]:
# 4. Modelo Propuesto: Random Forest (con balanceo de clases)
pipe_rf = Pipeline([
    ('prep', preprocesador),
    (
        'clf',
        RandomForestClassifier(
            n_estimators=150,
            max_depth=8,  # Evita sobreajuste (overfitting)
            class_weight='balanced',
            random_state=42,
        ),
    ),
])
pipe_rf.fit(X_train, y_train)
preds_rf = pipe_rf.predict(X_test)
probs_rf = pipe_rf.predict_proba(X_test)[:, 1]

In [ ]:
# 5. Comparativa de resultados
print('=== BASELINE: NAIVE BAYES ===')
print(
    f'F1-Score: {f1_score(y_test, preds_nb):.4f} | ROC-AUC:'
    f' {roc_auc_score(y_test, probs_nb):.4f}'
)

print('\n=== PROPUESTO: RANDOM FOREST ===')
print(
    f'F1-Score: {f1_score(y_test, preds_rf):.4f} | ROC-AUC:'
    f' {roc_auc_score(y_test, probs_rf):.4f}'
)
print('\nReporte detallado Random Forest:')
print(classification_report(y_test, preds_rf))

In [ ]:
# 6. Guardar el pipeline completo (preprocesador + modelo) para la app
joblib.dump(pipe_rf, 'modelo_cobranza_rf.pkl')
print("Modelo guardado como 'modelo_cobranza_rf.pkl'")